# Fase 3: Programación orientada a objetos

Se utilizará como referencia el dataset de F2, correspondiente a licitaciones adjudicadas y desiertas del primer semestre de 2026, el objetivo es construir componentes reutilizables y medir su eficiencia.

In [2]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd().resolve().parents[1]

carpeta_f2 = raiz / "F2" / "notebooks" / "resultados_f2"

print("Raíz del proyecto:", raiz)
print("¿Existe la carpeta de F2?:", carpeta_f2.is_dir())

archivos = [
    "licitaciones_filtradas_provisional.csv.gz",
    "matriz_filtrada_provisional.csv.gz",
    "registros_pendientes_revision.csv",
    "resumen_final_f2.csv"
]

for archivo in archivos:
    ruta = carpeta_f2 / archivo
    print(f"{archivo}: {ruta.is_file()}")

Raíz del proyecto: C:\Users\nacho\proyecto-grupo_1-mcdi500
¿Existe la carpeta de F2?: True
licitaciones_filtradas_provisional.csv.gz: True
matriz_filtrada_provisional.csv.gz: True
registros_pendientes_revision.csv: True
resumen_final_f2.csv: True


In [3]:

# Cargamos el conjunto analítico de F2
df_licitaciones_f2 = pd.read_csv(
    carpeta_f2 / "licitaciones_filtradas_provisional.csv.gz"
)

df_matriz_f2 = pd.read_csv(
    carpeta_f2 / "matriz_filtrada_provisional.csv.gz"
)

print("Licitaciones F2:", df_licitaciones_f2.shape)
print("Matriz F2:", df_matriz_f2.shape)
print("Códigos únicos:",
      df_licitaciones_f2["CodigoExterno"].nunique())
print("Códigos duplicados:",
      df_licitaciones_f2["CodigoExterno"].duplicated().sum())

print("\nDistribución de EsDesierta:")
print(df_licitaciones_f2["EsDesierta"].value_counts())

assert len(df_licitaciones_f2) == 41607
assert len(df_matriz_f2) == 41607
assert df_licitaciones_f2["CodigoExterno"].is_unique

print("\nValidación inicial de F3 correcta")

Licitaciones F2: (41607, 18)
Matriz F2: (41607, 40)
Códigos únicos: 41607
Códigos duplicados: 0

Distribución de EsDesierta:
EsDesierta
0    35622
1     5985
Name: count, dtype: int64

Validación inicial de F3 correcta


Despúes de la comprobación traeremos lo procesado en F2 mediante funciones independientes. Primero vamos a cargar los datos.

In [4]:

import sys
ruta_modulos = str(raiz / "F3" / "src")

if ruta_modulos not in sys.path:
    sys.path.insert(0, ruta_modulos)
    
from carga import cargar_datos

df_cargado = cargar_datos(
    carpeta_f2 / "licitaciones_filtradas_provisional.csv.gz"
)

# Con esto buscamos comprobamos que la nueva carga conserva el resultado
pd.testing.assert_frame_equal(
    df_cargado,
    df_licitaciones_f2
)

print("La función de carga conserva los datos de F2.")

Cargado: 41607 filas y 18 columnas
La función de carga conserva los datos de F2.


### Creación de la variable objetivo

Creamos una función para generar la variable binaria EsDesierta, con esta vamos distinguir entre licitaciones adjudicadas (0) y desiertas (1).

In [6]:
from preparacion import crear_variable_objetivo

df_entrada = df_licitaciones_f2.drop(
    columns=["EsDesierta"]
).copy()

df_objetivo = crear_variable_objetivo(df_entrada)

pd.testing.assert_series_equal(
    df_objetivo["EsDesierta"],
    df_licitaciones_f2["EsDesierta"]
)

print("La variable objetivo coincide con F2")

print("\nDistribución de licitaciones:")
print(df_objetivo["EsDesierta"].value_counts())

La variable objetivo coincide con F2

Distribución de licitaciones:
EsDesierta
0    35622
1     5985
Name: count, dtype: int64


### Cálculo de los días entre publicación y cierre de las licitacions

Se implementa una función para calcular la diferencia en días entre las fechas de publicación y cierre de cada licitación.

In [7]:
import importlib
import preparacion

importlib.reload(preparacion)

# Lalmamos a la función
from preparacion import calcular_dias_publicacion_cierre

df_entrada_fechas = df_licitaciones_f2.drop(
    columns=["DiasPublicacionCierre"]
).copy()

df_fechas = calcular_dias_publicacion_cierre(
    df_entrada_fechas
)

# Comparamos el resultado con F2 para saber que se conservan los rsultados
pd.testing.assert_series_equal(
    df_fechas["DiasPublicacionCierre"],
    df_licitaciones_f2["DiasPublicacionCierre"],
    check_exact=False,
    rtol=1e-10,
    atol=1e-10
)

print("El cálculo de días coincide con F2")

print("\nDuraciones negativas:",
      (df_fechas["DiasPublicacionCierre"] < 0).sum())

print("Duraciones faltantes:",
      df_fechas["DiasPublicacionCierre"].isna().sum())

print("\nPrimeros cinco resultados:")
display(
    df_fechas[
        ["CodigoExterno", "DiasPublicacionCierre"]
    ].head()
)

El cálculo de días coincide con F2

Duraciones negativas: 0
Duraciones faltantes: 0

Primeros cinco resultados:


,CodigoExterno,DiasPublicacionCierre
0,799512-11-LE25,14.0
1,2408-54-LP25,35.0
2,743848-1-LE25,26.0
3,1641-157-LR25,30.0
4,1058095-67-LE25,10.0


### Clase Transformador

Se implementa la base que establece una estructura común para las transformaciones de datos.

In [8]:
from transformadores import Transformador

transformador = Transformador("MontoEstimadoNumerico")

print("Columna seleccionada:", transformador.columna)
print("¿Está ajustado?:", transformador._ajustado)

try:
    transformador.transformar(df_licitaciones_f2)

except RuntimeError as error:
    print("Validación correcta:", error)

Columna seleccionada: MontoEstimadoNumerico
¿Está ajustado?: False
Validación correcta: Primero debe ejecutar ajustar()


### Clase ImputadorMediana

Se implementa la clase `ImputadorMediana`, que hereda transformador y permite tratar los valores faltantes de la variable MontoEstimadoNumerico.

In [9]:
import importlib
import transformadores

importlib.reload(transformadores)

from transformadores import ImputadorMediana

# Preparar los datos sin las columnas de imputación
df_entrada_montos = df_licitaciones_f2.drop(
    columns=[
        "MontoEstimadoImputado",
        "MontoFueImputado"
    ]
).copy()

# Crear y ajustar el imputador
imputador = ImputadorMediana("MontoEstimadoNumerico")

imputador.ajustar(df_entrada_montos)

# Aplicar la transformación
df_montos_f3 = imputador.transformar(
    df_entrada_montos
)

# Comparar los montos imputados con F2
pd.testing.assert_series_equal(
    df_montos_f3["MontoEstimadoImputado"],
    df_licitaciones_f2["MontoEstimadoImputado"]
)

# Comparar los indicadores de imputación con F2
pd.testing.assert_series_equal(
    df_montos_f3["MontoFueImputado"],
    df_licitaciones_f2["MontoFueImputado"]
)

print("La imputación coincide con F2")

print("\nMediana CLP utilizada:",
      imputador._parametros["mediana"])

print("Montos imputados:",
      df_montos_f3["MontoFueImputado"].sum())

print("Nulos restantes:",
      df_montos_f3["MontoEstimadoImputado"].isna().sum())

La imputación coincide con F2

Mediana CLP utilizada: 27600000.0
Montos imputados: 7
Nulos restantes: 0
